# Genre Bias in the Academy Awards: A Statistical Analysis of Oscar Winning Films

**Group Members:** Edison Ayran, Diego Inostroza

In [1]:
# libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency
from statsmodels.sandbox.stats.runs import runstest_1samp
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Introduction

This project investigates whether the genre of a film influences its probability of winning an Academy Award. The Academy Awards are widely regarded as one of the most prestigious honors in the film industry, and winning an Oscar can significantly impact a film’s cultural recognition, financial success, and the careers of those involved. Because of this influence, it is important to examine whether certain types of films are systematically favored in award outcomes. Using data from the Academy Awards database and the IMDb dataset, this analysis explores whether genres such as Drama or Biography which are often considered “prestige genres,” are more likely to win awards than genres such as Action, Comedy, or Horror. By combining award data with film metadata, this project aims to statistically evaluate whether genre plays a meaningful role in Oscar success.

# Problem Statement

Despite the widely held belief that the Academy Awards recognize the "best" films of a given year on purely artistic merit, critics and scholars have long argued that the selection process is subject to **systematic genre bias** so films belonging to certain genres are disproportionately likely to receive nominations and wins compared to others. Dramatic and biographical films, often labeled "prestige cinema," are seen to appear more among nominees and winners in major categories, while genres such as Comedy and Horror are seen as absent. This raises a striking question: Is genre an independent predictor of Oscar success, or are voters simply recognizing that films in prestige genres tend to be better received?

This project investigates whether a film's genre classification significantly influences its probability of winning an Academy Award across four **film-level award categories**, Best Picture, Directing, Original Screenplay, and Adapted Screenplay from **2004 to 2024**. We restrict our analysis to these four categories because they are the most directly attributable to a film as a whole (rather than to an individual performance or technical craft), making genre the most logically relevant explanatory variable.

We further ask whether any observed genre bias has **shifted over time**, particularly in light of structural changes to the Academy: the "Oscars So White" controversy (2015–2016), membership diversification efforts, and the emergence of streaming platforms as major content producers and award campaigners.

To construct our comparison group, we contrast verified Oscar winners against a random sample of films released in the same period that did not win an Academy Award. This design allows us to characterize what distinguishes winning films from the broader population of films released during our study window, with genre as the primary explanatory variable.

# Objective

Determine whether movie genre significantly influences the probability of winning an Academy
Award across four major film-level categories (Best Picture, Directing, Original Screenplay,
Adapted Screenplay) from 2004–2024, and test whether this relationship has changed
meaningfully between the pre-reform era (2004–2014) and the post-reform era (2015–2024)
using chi-squared tests of independence, the Wald-Wolfowitz runs test, and logistic
regression with genre indicator variables and genre × year interaction terms.

### Research Question

**Does a film's genre classification significantly influence its probability of winning an
Academy Award, and has this genre-based bias changed over the last two decades?**

More specifically: Are "prestige" genres such as Drama and Biography systematically
over-represented among winners relative to their share of nominations, and if so, has
the Academy's membership reform period (post-2015) produced a measurable reduction in
this effect?

This question allows us to investigate whether the Academy's definition of cinematic
excellence is genre-agnostic, or whether certain storytelling formats carry a persistent
structural advantage in the voting process.

# Description of Variables

**From `full_data.csv`:**

| Column              | Type    | Description |
|---------------------|---------|-------------|
| `FilmId`            | string  | IMDb tconst (e.g. `tt0081398`) - primary merge key with IMDb datasets |
| `Film`              | string  | Title of the nominated film |
| `Year`              | string  | Ceremony year (e.g. `2004/05`) |
| `CanonicalCategory` | string  | Standardized category name - filtered to our 4 target categories |
| `Category`          | string  | Raw category label as listed in the official Academy database |
| `Winner`            | boolean | True if the film won; NaN treated as False (nominated but did not win) |
| `Name`              | string  | Person associated with the nomination (director or writer) |

**From `title.basics.tsv.gz`:**

| Column           | Type   | Description |
|------------------|--------|-------------|
| `tconst`         | string | IMDb title identifier - merge key |
| `genres`         | string | Comma-separated genre tags, up to 3 per film (e.g. `Drama,Biography,History`) |
| `runtimeMinutes` | float  | Film runtime in minutes - regression control variable |
| `titleType`      | string | Used to pre-filter to movies only before merging |
| `startYear`      | int    | Release year - used to cross-validate ceremony year |

**From `title.ratings.tsv.gz`:**

| Column          | Type   | Description |
|-----------------|--------|-------------|
| `tconst`        | string | IMDb title identifier - merge key |
| `averageRating` | float  | Weighted average IMDb user rating (scale 1–10) |
| `numVotes`      | int    | Total number of user ratings received |

**Engineered during cleaning:**

| Column          | Type   | Description |
|-----------------|--------|-------------|
| `primary_genre` | string | First genre extracted from `genres` - core categorical predictor in all 3 tests |
| `outcome`       | int    | Binary re-encoding of `Winner` - 1 if won, 0 if nominated only |
| `era`           | int    | 0 = pre-reform (2004–2014), 1 = post-reform (2015–2024) |
| `log_votes`     | float  | Log-transformed `numVotes` to reduce right skew before regression |

## Dataset Composition

Our analysis uses three files merged via IMDb tconst identifiers.

**`full_data.csv` (Primary Oscar Source)**
Contains 12,014 nomination records across all Academy Award categories from 1927–2025.
Each record includes a `FilmId` column with an IMDb tconst (e.g. `tt0081398`),
enabling a clean exact join with IMDb
After filtering to our 4 target categories and ceremony years 2004–2024:

| Category                       | Nominations | Winners |
|--------------------------------|-------------|---------|
| BEST PICTURE                   | 171         | 21      |
| DIRECTING                      | 105         | 21      |
| WRITING (Original Screenplay)  | 105         | 21      |
| WRITING (Adapted Screenplay)   | 105         | 21      |
| **Total**                      | **486**     | **84**  |

**`title.basics.tsv.gz` (IMDb)**
Covers ~10 million IMDb titles. Provides genre classifications (up to 3 per film),
runtime in minutes, title type, and release year. Filtered to `titleType == 'movie'`
before merging.

**`title.ratings.tsv.gz` (IMDb)**
Provides weighted average user rating and total vote count for all rated titles.
Joined to basics on `tconst`, then merged with the Oscar records on `FilmId = tconst`.

Final working dataset (`combined_df`) used for all analysis: **~1,084 films** 
(deduplicated Oscar winners from the 486 nomination records, combined with 1,000 randomly 
sampled non-winning films), across 6 analytical columns.

## Load the Data ##

In [ ]:
# Full Oscar dataset 
oscars_raw = pd.read_csv('full_data.csv', sep='\t', low_memory=False)

# IMDb title.basics 
imdb_basics = pd.read_csv(
    'title.basics.tsv.gz',
    sep='\t',
    compression='gzip',
    low_memory=False,
    na_values='\\N'
)

# IMDb title.ratings 
imdb_ratings = pd.read_csv(
    'title.ratings.tsv.gz',
    sep='\t',
    compression='gzip',
    na_values='\\N'
)

print("Oscar records :", oscars_raw.shape)   
print("IMDb basics   :", imdb_basics.shape)
print("IMDb ratings  :", imdb_ratings.shape)

oscars_raw.head()

In [ ]:
imdb_basics.head()

In [ ]:
imdb_ratings.head()

## Filter to 4 Target Categories (edit)

In [ ]:
target_categories = [
    'BEST PICTURE',
    'DIRECTING',
    'WRITING (Original Screenplay)',
    'WRITING (Adapted Screenplay)'
]

oscars_filtered = oscars_raw[oscars_raw['CanonicalCategory'].isin(target_categories)].copy()
print(f"After category filter: {len(oscars_filtered)} records")

oscars_filtered['year_numeric'] = oscars_filtered['Year'].str.split('/').str[0].astype(int)

oscars_filtered = oscars_filtered[
    (oscars_filtered['year_numeric'] >= 2004) & 
    (oscars_filtered['year_numeric'] <= 2024)
].copy()
print(f"After 2004-2024 filter: {len(oscars_filtered)} records")

print("\nNominations by category:")
print(oscars_filtered['CanonicalCategory'].value_counts())
print()

## Merge with IMDb Data (Genres + Ratings) (edit)

In [ ]:
oscars_with_genre = oscars_filtered.merge(
    imdb_basics[['tconst', 'genres', 'runtimeMinutes', 'startYear']],
    left_on='FilmId',
    right_on='tconst',
    how='left'
)
print(f"After genre merge: {len(oscars_with_genre)} records")
print(f"Missing genres: {oscars_with_genre['genres'].isna().sum()}")

oscars_with_genre = oscars_with_genre.merge(
    imdb_ratings[['tconst', 'averageRating', 'numVotes']],
    on='tconst',
    how='left'
)
print(f"After ratings merge: {len(oscars_with_genre)} records")
print(f"Missing ratings: {oscars_with_genre['averageRating'].isna().sum()}")
print()

## Cleaning Dataset (edit)

In [ ]:
df = oscars_with_genre.dropna(subset=['genres']).copy()

print(f"After dropping missing genres: {len(df)} records")

df['outcome'] = df['Winner'].apply(
    lambda x: 1 if x == True or x == 'True' else 0
)

df['primary_genre'] = df['genres'].str.split(',').str[0]

df['era'] = df['year_numeric'].apply(
    lambda x: 0 if x < 2015 else 1
)

df['runtimeMinutes'] = pd.to_numeric(df['runtimeMinutes'], errors='coerce')
df['averageRating'] = pd.to_numeric(df['averageRating'], errors='coerce')
df['numVotes'] = pd.to_numeric(df['numVotes'], errors='coerce')

df['log_votes'] = np.log1p(df['numVotes'].fillna(0))

final_df = df[[
    'year_numeric',
    'Film',
    'CanonicalCategory',
    'Category',
    'outcome',
    'primary_genre',
    'genres',
    'averageRating',
    'numVotes',
    'log_votes',
    'runtimeMinutes',
    'era',
    'FilmId'
]].copy()

final_df.rename(columns={
    'year_numeric': 'Year',
    'CanonicalCategory': 'Award_Category'
}, inplace=True)

final_df = final_df.sort_values(['Year', 'Award_Category']).reset_index(drop=True)

print(f"Final dataset: {len(final_df)} nominations")
print()


## Initial dataset exploration (edit)

In [ ]:
print("Dataset Overview")
print("="*80)
print()

print(f"Total records: {len(final_df)}")
print(f"Number of columns: {len(final_df.columns)}")
print()

print("Columns:")
print(final_df.columns.tolist())
print()

print("Award Outcome Distribution:")
print(final_df['outcome'].value_counts())
print()

print("Award Categories:")
print(final_df['Award_Category'].value_counts())
print()

print(f"Year range: {final_df['Year'].min()} to {final_df['Year'].max()}")
print()

print("Sample of dataset:")
print(final_df.head(10))
print()

print("Genre Distribution (Top 15):")
print(final_df['primary_genre'].value_counts().head(15))

## Merge the Oscars Dataset ##

In [ ]:
winning_films = oscars_raw[oscars_raw['Winner'] == True][oscars_raw['Class'].isin(['Title', 'Writing'])]
winning_films
winning_films_with_genre = winning_films.merge(
    imdb_basics[['tconst', 'genres']],
    left_on='FilmId',
    right_on='tconst',
    how='left'
)
winning_films_with_genre = winning_films_with_genre.merge(
    imdb_ratings[['tconst', 'averageRating']],
    on='tconst',
    how='left'
)
winning_films_with_genre

## Filter and Clean the Merged Dataset ##

In [ ]:
winning_films_in_range = winning_films_with_genre[~winning_films_with_genre['Year'].isin(['1927/28', '1928/29', '1929/30', '1930/31', '1931/32', '1932/33'])]
winning_films_in_range = winning_films_in_range[(winning_films_in_range['Year'].astype(int) >= 2004) & (winning_films_in_range['Year'].astype(int) <= 2024)]

cleaned_winners = winning_films_in_range.drop_duplicates(subset=['Film'], keep='first').reset_index()[['Year', 'Film', 'genres', 'averageRating', 'tconst']]
cleaned_winners = cleaned_winners.dropna()
cleaned_winners

## Filter and Randomly Sample Non-Winners ##

In [ ]:
filtered_imdb = imdb_basics[~imdb_basics['tconst'].isin(cleaned_winners['tconst'].tolist())]
filtered_imdb = filtered_imdb[(filtered_imdb['startYear'] >= 2004) & (filtered_imdb['startYear'] <= 2024)]
filtered_imdb = filtered_imdb[filtered_imdb['titleType'].isin(['movie', 'short'])]

filtered_imdb = filtered_imdb.merge(
    imdb_ratings[['tconst', 'averageRating']],
    on='tconst',
    how='left'
)

filtered_imdb = filtered_imdb[~filtered_imdb['averageRating'].isna()]
filtered_imdb

In [ ]:
random_nonwinners = filtered_imdb.sample(n=1000, replace=False)
random_nonwinners = random_nonwinners.reset_index()
random_nonwinners

In [ ]:
random_nonwinners = random_nonwinners[['startYear', 'primaryTitle', 'genres', 'averageRating', 'tconst']]
random_nonwinners['startYear'] = random_nonwinners['startYear'].astype(int)
random_nonwinners.columns = ['Year', 'Film', 'genres', 'averageRating', 'tconst']
random_nonwinners

In [ ]:
cleaned_winners['winner'] = [True] * cleaned_winners.shape[0]
random_nonwinners['winner'] = [False] * random_nonwinners.shape[0]

combined_df = pd.concat([cleaned_winners, random_nonwinners])
combined_df['genres'] = combined_df['genres'].str.split(',')
combined_df = combined_df.reset_index(drop=True)
combined_df

# **Exploratory Data Analysis (EDA)**

# EDA set-up and Overview

In [ ]:
print("=== combined_df shape ===")
print(combined_df.shape)
print()
print("=== Winner breakdown ===")
print(combined_df['winner'].value_counts())
print()
print("=== Year range ===")
years = combined_df['Year'].astype(int)
print(f"Min year: {years.min()}  |  Max year: {years.max()}")
print()
print("=== Missing genre values ===")
print(f"Rows missing genre: {combined_df['genres'].isna().sum()}")
print()
combined_df.head(10)

# Primary Genre Extraction

In [ ]:
combined_df['primary_genre'] = combined_df['genres'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

eda_df = combined_df.dropna(subset=['primary_genre']).copy()

print("=== Primary genre value counts (top 15) ===")
print(eda_df['primary_genre'].value_counts().head(15))

# Genre Distribution (All Films)

In [ ]:
genre_counts = eda_df['primary_genre'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(genre_counts.index, genre_counts.values, color='steelblue', edgecolor='white')
ax.set_title('Genre Distribution Among All Films in Dataset (2004–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Primary Genre')
ax.set_ylabel('Number of Films')
ax.tick_params(axis='x', rotation=30)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

# Genre Distribution (Winners Only)

In [ ]:
winners_df = eda_df[eda_df['winner'] == True]
winner_genre_counts = winners_df['primary_genre'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(winner_genre_counts.index, winner_genre_counts.values, color='goldenrod', edgecolor='white')
ax.set_title('Genre Distribution Among Oscar Winners (2004–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Primary Genre')
ax.set_ylabel('Number of Winning Films')
ax.tick_params(axis='x', rotation=30)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

# Win Rate by Genre

In [ ]:
genre_totals = eda_df.groupby('primary_genre').size()
genre_wins   = eda_df[eda_df['winner'] == True].groupby('primary_genre').size()

win_rate = (genre_wins / genre_totals).dropna().sort_values(ascending=False)
win_rate = win_rate[genre_totals[win_rate.index] >= 5]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['goldenrod' if g == 'Drama' else 'steelblue' for g in win_rate.index]
bars = ax.bar(win_rate.index, win_rate.values * 100, color=colors, edgecolor='white')
ax.set_title('Win Rate by Primary Genre (min. 5 films, 2004–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Primary Genre')
ax.set_ylabel('Win Rate (%)')
ax.tick_params(axis='x', rotation=35)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
plt.tight_layout()
plt.show()

print(win_rate.apply(lambda x: f"{x*100:.1f}%"))

# Stacked Comparison: Winners vs Non-Winners by Genre 

In [ ]:
top_genres = genre_totals.nlargest(10).index
plot_data  = eda_df[eda_df['primary_genre'].isin(top_genres)].copy()
ct = plot_data.groupby(['primary_genre', 'winner']).size().unstack(fill_value=0)
ct.columns = ['Non-Winner', 'Winner']
ct = ct.loc[top_genres]

ct.plot(
    kind='bar',
    stacked=True,
    color=['#4878CF', '#C44E52'],
    figsize=(11, 5),
    edgecolor='white'
)
plt.title('Winner vs. Non-Winner Counts by Genre (Top 10, 2004–2024)', fontsize=14, fontweight='bold')
plt.xlabel('Primary Genre')
plt.ylabel('Number of Films')
plt.xticks(rotation=30)
plt.legend(title='Outcome')
plt.tight_layout()
plt.show()

# Temporal Trend: Drama Share of Winners by Period 


In [ ]:
eda_df['Period'] = pd.cut(
    eda_df['Year'].astype(int),
    bins=[2003, 2009, 2014, 2019, 2024],
    labels=['2004–2009', '2010–2014', '2015–2019', '2020–2024']
)

period_wins = eda_df[eda_df['winner'] == True].groupby(
    ['Period', 'primary_genre'], observed=True
).size().unstack(fill_value=0)

drama_share = (period_wins.get('Drama', 0) / period_wins.sum(axis=1) * 100).dropna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(drama_share.index.astype(str), drama_share.values, marker='o', color='goldenrod', linewidth=2.5)
ax.set_title("Drama's Share of Oscar Winners by 5-Year Period (2004–2024)", fontsize=13, fontweight='bold')
ax.set_xlabel('Period')
ax.set_ylabel('Drama Win Share (%)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_ylim(0, 100)
for x, y in zip(drama_share.index.astype(str), drama_share.values):
    ax.annotate(f"{y:.0f}%", (x, y), textcoords="offset points", xytext=(0, 8), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

# IMDb Rating Distribution: Winners vs Non-Winners 


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for label, color, subset in [
    ('Non-Winner', '#4878CF', eda_df[eda_df['winner'] == False]),
    ('Winner',     '#C44E52', eda_df[eda_df['winner'] == True])
]:
    subset['averageRating'].dropna().plot.kde(ax=ax, label=label, color=color, linewidth=2)

ax.set_title('IMDb Rating Distribution: Winners vs. Non-Winners (2004–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('IMDb Average Rating')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

print("=== Mean IMDb Rating by Outcome ===")
print(eda_df.groupby('winner')['averageRating'].describe().round(2))

## EDA Summary

The exploratory analysis reveals several patterns that motivate our formal hypothesis testing:

1. **Drama dominates among winners.** Drama is the most prevalent primary genre among Oscar-winning films across 2004–2024. The win-rate plot confirms that Drama films not only appear most frequently but also convert at a higher rate relative to other genres present in the dataset.

2. **Comedy, Horror, and Action are structurally underrepresented.** Films classified primarily under these genres appear rarely among winners, and the stacked bar chart shows this disparity is not simply a function of fewer such films existing, they are present in the non-winner pool but largely absent from the winner group.

3. **The Drama advantage has not clearly declined over time.** The temporal trend plot shows that Drama's share of winners fluctuates across five-year periods but does not exhibit a sustained downward trend, suggesting that changes in Academy membership have not fully eliminated the prestige-genre advantage through 2024.

4. **Winners receive moderately higher IMDb ratings.** The rating distribution plot shows Oscar-winning films cluster slightly higher on IMDb scores compared to non-winners, confirming that raw quality (as represented by user ratings) partially confounds the genre-outcome relationship. This motivates the inclusion of `averageRating` as a control variable in our logistic regression model, where the binary outcome (winner vs. non-winner) is the dependent variable being predicted from genre indicators and film-level covariates.

## One Hot Encode the Genres ##

In [ ]:
one_hot = pd.get_dummies(combined_df.explode('genres'), columns=['genres'])
one_hot = one_hot.groupby('tconst').max().reset_index(drop=True)
one_hot